%md
%md
# Petroleum Pipeline — Shared Core

## 1. Configure Shared Dependencies
Provide reusable Bronze and Silver pipeline functions for
end-to-end execution.

This notebook contains setup and function definitions.
It does not execute data loads, API requests, or controlled tests.

In [0]:
import hashlib
import json
import math

from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
    DoubleType,
    DateType,
    TimestampType,
    ArrayType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

table_prefix = "workspace.default"

bronze_table = f"{table_prefix}.petroleum_bronze"
silver_table = f"{table_prefix}.petroleum_silver"
quarantine_table = f"{table_prefix}.petroleum_quarantine"
execution_log_table = f"{table_prefix}.petroleum_execution_logs"

required_tables = [
    bronze_table,
    silver_table,
    quarantine_table,
    execution_log_table,
]

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Required Phase 2 table is missing: {table_name}")

print("Shared pipeline dependencies ready.")

%md
%md
## 2. Configure the Raw Contract and Shared Table Schemas
Define the raw observation schema explicitly.
Map source field names to Bronze column names.
Reuse the schemas of the existing explicitly defined Delta tables.

In [0]:
# Explicit source contract: raw observation values are strings.
observation_schema = StructType([
    StructField("period", StringType(), True),
    StructField("duoarea", StringType(), True),
    StructField("area-name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("product-name", StringType(), True),
    StructField("process", StringType(), True),
    StructField("process-name", StringType(), True),
    StructField("series", StringType(), True),
    StructField("series-description", StringType(), True),
    StructField("value", StringType(), True),
    StructField("units", StringType(), True),
])

expected_fields = set(observation_schema.fieldNames())
required_key_fields = {"period", "series"}

# Preserve the source-to-Bronze naming convention.
bronze_field_mapping = {
    "period": "period",
    "duoarea": "duoarea",
    "area-name": "area_name",
    "product": "product",
    "product-name": "product_name",
    "process": "process",
    "process-name": "process_name",
    "series": "series",
    "series-description": "series_description",
    "value": "value",
    "units": "units",
}

# These tables were created with explicit schemas in Phase 2.
bronze_schema = spark.table(bronze_table).schema
silver_schema = spark.table(silver_table).schema
quarantine_schema = spark.table(quarantine_table).schema

# Include the optional Gold metrics added to the shared audit table.
execution_log_schema = spark.table(execution_log_table).schema

silver_key_columns = ["period", "series"]

print(f"Raw contract fields: {len(expected_fields)}")
print(f"Bronze schema columns: {len(bronze_schema.fields)}")
print(f"Silver schema columns: {len(silver_schema.fields)}")
print("Shared schemas and field mapping ready.")

%md
%md
## 3. Define Shared Logging and Raw Validation
Record pipeline execution outcomes in the shared audit table.
Validate raw field names, value types, and required observation keys.

In [0]:
def write_execution_log(
    *,
    run_id,
    batch_id,
    load_type,
    layer,
    source_reference,
    target_table,
    start_time,
    end_time,
    status,
    rows_read=None,
    rows_inserted=None,
    rows_updated=None,
    rows_quarantined=None,
    error_message=None,
):
    """Append one Bronze or Silver operation to the shared audit table."""
    if status not in {"Success", "Failure"}:
        raise ValueError("Status must be Success or Failure.")

    # Read the current schema so optional Gold columns are supported.
    log_schema = spark.table(execution_log_table).schema
    log_record = {field.name: None for field in log_schema.fields}

    log_record.update({
        "log_id": str(uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "load_type": load_type,
        "layer": layer,
        "source_reference": str(source_reference),
        "target_table": target_table,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": rows_quarantined,
        "error_message": error_message,
        "load_timestamp": datetime.now(timezone.utc),
    })

    (
        spark.createDataFrame([log_record], schema=log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(execution_log_table)
    )


def validate_raw_record(record):
    """Return rejection reasons; an empty list means the record is valid."""
    if not isinstance(record, dict):
        return ["Observation must be a JSON object."]

    reasons = []
    actual_fields = set(record.keys())
    expected_fields = set(observation_schema.fieldNames())
    required_key_fields = {"period", "series"}

    extra_fields = sorted(actual_fields - expected_fields)
    missing_fields = sorted(expected_fields - actual_fields)

    if extra_fields:
        reasons.append(
            f"Unexpected fields: {', '.join(extra_fields)}"
        )

    if missing_fields:
        reasons.append(
            f"Missing fields: {', '.join(missing_fields)}"
        )

    for field_name in sorted(actual_fields & expected_fields):
        value = record[field_name]

        if value is not None and not isinstance(value, str):
            reasons.append(
                f"Invalid type for {field_name}: expected string, "
                f"received {type(value).__name__}"
            )

        if field_name in required_key_fields:
            if value is None or (
                isinstance(value, str) and not value.strip()
            ):
                reasons.append(
                    f"Empty observation key: {field_name}"
                )

    return reasons


print("Shared execution logging function ready.")
print("Raw record validation function ready.")

%md
%md
## 4. Define File Validation and Quarantine Writing
Separate accepted and rejected observations from each raw file.
Preserve rejected records with their reasons and ingestion metadata.

In [0]:
def validate_raw_file(file_path):
    """Validate one raw file and return accepted and rejected records."""
    file_path = Path(file_path)

    with file_path.open("r", encoding="utf-8") as source:
        payload = json.load(source)

    if not isinstance(payload, dict):
        raise ValueError("The file must contain a JSON object.")

    response = payload.get("response", payload)

    if not isinstance(response, dict):
        raise ValueError("The response must be a JSON object.")

    observations = response.get("data")

    if not isinstance(observations, list):
        raise ValueError("The data field must contain a JSON array.")

    accepted_records = []
    rejected_records = []

    for record_index, record in enumerate(observations):
        reasons = validate_raw_record(record)

        if reasons:
            rejected_records.append({
                "source_file": str(file_path),
                "record_index": record_index,
                "raw_payload": json.dumps(record, ensure_ascii=False),
                "rejection_reason": "; ".join(reasons),
            })
        else:
            accepted_records.append({
                "record_index": record_index,
                "observation": record,
            })

    return accepted_records, rejected_records


def write_quarantine_records(
    rejected_records, *, run_id, batch_id, load_type, layer
):
    """Save rejected records for one processing attempt."""
    if not rejected_records:
        return 0

    processed_at = datetime.now(timezone.utc)

    quarantine_rows = [
        {
            "quarantine_id": str(uuid4()),
            "run_id": run_id,
            "batch_id": batch_id,
            "load_type": load_type,
            "layer": layer,
            "source_file": record["source_file"],
            "record_index": record["record_index"],
            "raw_payload": record["raw_payload"],
            "rejection_reason": record["rejection_reason"],
            "load_timestamp": processed_at,
        }
        for record in rejected_records
    ]

    quarantine_df = spark.createDataFrame(
        quarantine_rows,
        schema=quarantine_schema,
    )

    (
        quarantine_df.write.format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )

    return len(quarantine_rows)


print("Raw file validation function ready.")
print("Quarantine writer ready.")

%md
%md
## 5. Prepare Bronze Records and Stable Identifiers
Preserve the Phase 2 hashing logic for compatible reruns.
Map source fields to Bronze columns and attach ingestion metadata.
Exclude run IDs and timestamps from record identity.

In [0]:
def stable_hash(value):
    """Generate a deterministic SHA-256 hash from JSON-compatible data."""
    serialized = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


def prepare_bronze_records(
    accepted_records,
    *,
    source_file,
    batch_id,
    load_type,
    run_id,
    load_timestamp,
):
    """Prepare validated observations using the Bronze data contract."""
    bronze_rows = []
    source_file = str(source_file)

    for accepted_record in accepted_records:
        observation = accepted_record["observation"]
        record_index = accepted_record["record_index"]
        record_hash = stable_hash(observation)

        # Keep the same identity when retrying the same source input.
        bronze_id = stable_hash({
            "source_system": "EIA",
            "batch_id": batch_id,
            "load_type": load_type,
            "source_file": source_file,
            "record_index": record_index,
            "record_hash": record_hash,
        })

        bronze_row = {
            target_field: observation[source_field]
            for source_field, target_field in bronze_field_mapping.items()
        }

        bronze_row.update({
            "bronze_id": bronze_id,
            "record_hash": record_hash,
            "source_system": "EIA",
            "source_file": source_file,
            "record_index": record_index,
            "batch_id": batch_id,
            "load_type": load_type,
            "run_id": run_id,
            "load_timestamp": load_timestamp,
        })

        bronze_rows.append(bronze_row)

    return bronze_rows


print("Stable hashing function ready.")
print("Bronze record preparation function ready.")

%md
%md
## 6. Define Audited Raw-to-Bronze Ingestion
Validate one raw file, quarantine rejected observations, and insert
new Bronze records using stable identifiers.
Log successful and failed processing attempts.

In [0]:
def ingest_raw_file_to_bronze(
    file_path,
    *,
    batch_id,
    load_type,
    bronze_table=bronze_table,
):
    """Process one raw file with idempotent Bronze insertion and auditing."""
    file_path = Path(file_path)
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    view_name = f"bronze_source_{uuid4().hex}"

    rows_read = None
    rows_inserted = 0
    rows_quarantined = 0
    status = "Failure"
    error_message = None

    try:
        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        accepted_records, rejected_records = validate_raw_file(file_path)

        rows_read = len(accepted_records) + len(rejected_records)

        rows_quarantined = None
        rows_quarantined = write_quarantine_records(
            rejected_records,
            run_id=run_id,
            batch_id=batch_id,
            load_type=load_type,
            layer="Raw-to-Bronze",
        )

        if accepted_records:
            bronze_rows = prepare_bronze_records(
                accepted_records,
                source_file=file_path,
                batch_id=batch_id,
                load_type=load_type,
                run_id=run_id,
                load_timestamp=start_time,
            )

            source_df = (
                spark.createDataFrame(
                    bronze_rows,
                    schema=bronze_schema,
                )
                .dropDuplicates(["bronze_id"])
            )

            source_df.createOrReplaceTempView(view_name)

            # Leave the count unknown if MERGE or metric retrieval fails.
            rows_inserted = None

            merge_result = spark.sql(f"""
                MERGE INTO {bronze_table} AS target
                USING {view_name} AS source
                ON target.bronze_id = source.bronze_id

                WHEN NOT MATCHED
                THEN INSERT *
            """).collect()

            if len(merge_result) != 1:
                raise RuntimeError("MERGE did not return operation metrics.")

            metrics = merge_result[0].asDict()

            if "num_inserted_rows" not in metrics:
                raise RuntimeError("Inserted-row metric is unavailable.")

            rows_inserted = int(metrics["num_inserted_rows"])

        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                layer="Raw-to-Bronze",
                source_reference=file_path,
                target_table=bronze_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_inserted=rows_inserted,
                rows_updated=0,
                rows_quarantined=rows_quarantined,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": 0,
        "rows_quarantined": rows_quarantined,
    }


print("Audited raw-to-Bronze ingestion function ready.")

%md
%md
## 7. Define Parameterized Raw Batch Ingestion
Process all JSON pages in the selected folder.
Continue processing other files after individual failures.
Raise a batch-level error if any file fails, preventing downstream execution.

In [0]:
def ingest_raw_batch_to_bronze(
    folder_path,
    *,
    batch_id,
    load_type,
    bronze_table=bronze_table,
):
    """Process a selected raw folder and report file-level outcomes."""
    folder_path = Path(folder_path)

    if load_type not in {"full_load", "incremental_load"}:
        raise ValueError("Unsupported load type.")

    if not isinstance(batch_id, str) or not batch_id.strip():
        raise ValueError("Batch ID must be a nonempty string.")

    if not folder_path.is_dir():
        raise FileNotFoundError(f"Source folder not found: {folder_path}")

    files = sorted(folder_path.glob("page_*.json"))

    if not files:
        raise FileNotFoundError(f"No source files found: {folder_path}")

    successful_results = []
    failed_files = []

    for file_number, file_path in enumerate(files, start=1):
        try:
            result = ingest_raw_file_to_bronze(
                file_path,
                batch_id=batch_id,
                load_type=load_type,
                bronze_table=bronze_table,
            )
            successful_results.append(result)

            print(
                f"[{file_number}/{len(files)}] {file_path.name}: "
                f"inserted={result['rows_inserted']:,}, "
                f"quarantined={result['rows_quarantined']:,}"
            )

        except Exception as exc:
            failed_files.append({
                "source_file": str(file_path),
                "error": f"{type(exc).__name__}: {exc}",
            })

            print(
                f"[{file_number}/{len(files)}] "
                f"{file_path.name}: FAILED — {exc}"
            )

    # Row totals cover successful file operations only.
    summary = {
        "files_found": len(files),
        "files_succeeded": len(successful_results),
        "files_failed": len(failed_files),
        "rows_read": sum(r["rows_read"] for r in successful_results),
        "rows_inserted": sum(
            r["rows_inserted"] for r in successful_results
        ),
        "rows_updated": sum(
            r["rows_updated"] for r in successful_results
        ),
        "rows_quarantined": sum(
            r["rows_quarantined"] for r in successful_results
        ),
    }

    print("\nBatch summary")
    for metric, value in summary.items():
        print(f"{metric}: {value:,}")

    if failed_files:
        raise RuntimeError(
            f"{len(failed_files)} file(s) failed. "
            "Review file errors and audit logs before retrying. "
            "Summary row counts cover successful files only."
        )

    return summary


print("Parameterized raw batch ingestion function ready.")

%md
%md
## 8. Prepare Cleaned Silver Records
Convert monthly periods to dates and numeric strings to doubles.
Separate invalid records and preserve source metadata.
Generate a business-content hash for change detection.

In [0]:
def prepare_silver_dataframes(bronze_df, *, run_id, processed_at):
    """Return cleaned Silver candidates and rejected Bronze records."""
    checked_df = (
        bronze_df
        .withColumn("_period_text", F.trim(F.col("period")))
        .withColumn("_series_text", F.trim(F.col("series")))
        .withColumn(
            "_parsed_period",
            F.expr("try_cast(concat(_period_text, '-01') AS DATE)"),
        )
        .withColumn(
            "_parsed_value",
            F.expr("try_cast(trim(value) AS DOUBLE)"),
        )
    )

    invalid_period = (
        F.col("_period_text").isNull()
        | ~F.col("_period_text").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
        | F.col("_parsed_period").isNull()
    )

    invalid_series = (
        F.col("_series_text").isNull()
        | (F.col("_series_text") == "")
    )

    # Preserve source nulls, but reject invalid non-null numeric values.
    invalid_value = (
        F.col("value").isNotNull()
        & (
            F.col("_parsed_value").isNull()
            | F.isnan(F.col("_parsed_value"))
            | (
                F.abs(F.col("_parsed_value"))
                == F.lit(float("inf"))
            )
        )
    )

    checked_df = checked_df.withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                invalid_period,
                F.lit("Invalid monthly period"),
            ),
            F.when(
                invalid_series,
                F.lit("Missing series identifier"),
            ),
            F.when(
                invalid_value,
                F.lit("Invalid numeric value"),
            ),
        ),
    )

    rejected_df = (
        checked_df
        .filter(F.col("rejection_reason") != "")
        .select(*bronze_df.columns, "rejection_reason")
    )

    valid_df = checked_df.filter(F.col("rejection_reason") == "")

    text_columns = [
        "duoarea",
        "area_name",
        "product",
        "product_name",
        "process",
        "process_name",
        "series_description",
        "units",
    ]

    for column_name in text_columns:
        valid_df = valid_df.withColumn(
            column_name,
            F.trim(F.col(column_name)),
        )

    valid_df = (
        valid_df
        .withColumn("period", F.col("_parsed_period"))
        .withColumn("series", F.col("_series_text"))
        .withColumn("value", F.col("_parsed_value"))
        .withColumn(
            "bronze_load_timestamp",
            F.col("load_timestamp"),
        )
        .withColumn("run_id", F.lit(run_id))
        .withColumn(
            "load_timestamp",
            F.lit(processed_at).cast("timestamp"),
        )
    )

    # Preserve the Phase 2 field order and hashing convention.
    business_columns = list(bronze_field_mapping.values())

    valid_df = valid_df.withColumn(
        "record_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    *[F.col(name) for name in business_columns]
                ),
                options={"ignoreNullFields": "false"},
            ),
            256,
        ),
    )

    return (
        valid_df.select(*silver_schema.fieldNames()),
        rejected_df,
    )


print("Silver cleaning and validation function ready.")

%md
%md
## 9. Resolve Silver Duplicates and Conflicts
Keep one identical observation per period and series.
Separate all conflicting versions within the selected batch
for quarantine instead of choosing an arbitrary value.

In [0]:
def resolve_silver_duplicates(valid_df, bronze_df):
    """Deduplicate valid rows and return conflicting source records."""
    key_columns = ["period", "series"]

    # Different business hashes indicate conflicting observation versions.
    conflicting_keys_df = (
        valid_df.groupBy(*key_columns)
        .agg(
            F.countDistinct("record_hash").alias("version_count")
        )
        .filter(F.col("version_count") > 1)
        .select(*key_columns)
    )

    conflicting_ids_df = (
        valid_df
        .join(conflicting_keys_df, key_columns, "inner")
        .select("bronze_id")
        .distinct()
    )

    # Preserve original Bronze fields for quarantine.
    conflicting_records_df = (
        bronze_df
        .join(conflicting_ids_df, "bronze_id", "left_semi")
        .withColumn(
            "rejection_reason",
            F.lit(
                "Conflicting observations for the same period "
                "and series within the selected batch"
            ),
        )
    )

    non_conflicting_df = valid_df.join(
        conflicting_keys_df,
        key_columns,
        "left_anti",
    )

    # Select a consistent representative among identical observations.
    duplicate_window = (
        Window.partitionBy(*key_columns)
        .orderBy(
            F.col("bronze_load_timestamp").asc(),
            F.col("bronze_id").asc(),
        )
    )

    deduplicated_df = (
        non_conflicting_df
        .withColumn(
            "_duplicate_rank",
            F.row_number().over(duplicate_window),
        )
        .filter(F.col("_duplicate_rank") == 1)
        .drop("_duplicate_rank")
        .select(*silver_schema.fieldNames())
    )

    return deduplicated_df, conflicting_records_df


print("Silver duplicate and conflict handling function ready.")

%md
%md
## 10. Define Silver Quarantine Writing
Save rejected Bronze observations with their original source fields,
rejection reasons, and Silver processing metadata.

In [0]:
def write_silver_quarantine(rejected_df, *, run_id):
    """Save Silver-stage rejected records and return their count."""
    rejection_count = rejected_df.count()

    if rejection_count == 0:
        return 0

    # Reconstruct the source payload using original field names.
    raw_payload = F.to_json(
        F.struct(
            *[
                F.col(bronze_field).alias(source_field)
                for source_field, bronze_field
                in bronze_field_mapping.items()
            ]
        ),
        options={"ignoreNullFields": "false"},
    )

    quarantine_df = rejected_df.select(
        F.expr("uuid()").alias("quarantine_id"),
        F.lit(run_id).alias("run_id"),
        F.col("batch_id"),
        F.col("load_type"),
        F.lit("Bronze-to-Silver").alias("layer"),
        F.col("source_file"),
        F.col("record_index"),
        raw_payload.alias("raw_payload"),
        F.col("rejection_reason"),
        F.lit(datetime.now(timezone.utc))
        .cast("timestamp")
        .alias("load_timestamp"),
    )

    (
        quarantine_df
        .select(*quarantine_schema.fieldNames())
        .write.format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )

    return rejection_count


print("Silver quarantine writer ready.")

%md
%md
## 11. Define Audited Bronze-to-Silver Processing
Process a selected Bronze batch using cleaning, duplicate handling,
quarantine, and MERGE.

Normal mode inserts new observations and updates changed content.
Backfill mode inserts missing observations while preserving existing ones.
Log execution outcomes and row metrics.

In [0]:
def process_bronze_to_silver(
    *,
    batch_id,
    load_type,
    bronze_table=bronze_table,
    silver_table=silver_table,
    backfill_mode=False,
):
    """Process a Bronze batch with auditing and insert-only backfill."""
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    view_name = f"silver_source_{uuid4().hex}"

    source_reference = (
        f"{bronze_table}; batch_id={batch_id}; "
        f"load_type={load_type}; backfill_mode={backfill_mode}"
    )

    rows_read = None
    rows_inserted = 0
    rows_updated = 0
    rows_quarantined = 0
    status = "Failure"
    error_message = None

    try:
        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        if not isinstance(backfill_mode, bool):
            raise ValueError("backfill_mode must be True or False.")

        bronze_batch_df = (
            spark.table(bronze_table)
            .filter(
                (F.col("batch_id") == batch_id)
                & (F.col("load_type") == load_type)
            )
        )

        rows_read = bronze_batch_df.count()

        if rows_read == 0:
            raise ValueError("No Bronze records match the selected batch.")

        valid_df, invalid_df = prepare_silver_dataframes(
            bronze_batch_df,
            run_id=run_id,
            processed_at=start_time,
        )

        merge_df, conflicting_df = resolve_silver_duplicates(
            valid_df,
            bronze_batch_df,
        )

        rejected_df = invalid_df.unionByName(conflicting_df)

        rows_quarantined = None
        rows_quarantined = write_silver_quarantine(
            rejected_df,
            run_id=run_id,
        )

        merge_df.createOrReplaceTempView(view_name)

        # Historical backfills must not overwrite existing observations.
        update_clause = ""

        if not backfill_mode:
            update_clause = """
                WHEN MATCHED
                    AND NOT (target.record_hash <=> source.record_hash)
                THEN UPDATE SET *
            """

        rows_inserted = None
        rows_updated = None

        merge_result = spark.sql(f"""
            MERGE INTO {silver_table} AS target
            USING {view_name} AS source
            ON target.period = source.period
               AND target.series = source.series

            {update_clause}

            WHEN NOT MATCHED THEN INSERT *
        """).collect()

        if len(merge_result) != 1:
            raise RuntimeError("MERGE did not return operation metrics.")

        metrics = merge_result[0].asDict()

        if not {"num_inserted_rows", "num_updated_rows"}.issubset(metrics):
            raise RuntimeError("Required MERGE metrics are unavailable.")

        rows_inserted = int(metrics["num_inserted_rows"])
        rows_updated = int(metrics["num_updated_rows"])
        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                layer="Bronze-to-Silver",
                source_reference=source_reference,
                target_table=silver_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_inserted=rows_inserted,
                rows_updated=rows_updated,
                rows_quarantined=rows_quarantined,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": rows_quarantined,
    }


print("Audited Bronze-to-Silver processing function ready.")